# Etapa 5 — Redes Bayesianas

**DAG com 6 nós:**

```
     PORTE (população)        FRONT (UF faz fronteira com outro país)
              \                 /
               FACCAO (presença de facção, 2024)
             /          |           \
          TXH          ESC          TXT
  (taxa homicídio) (morte a esclarecer) (taxa tentativas)
```

**Justificativa das arestas**
- `PORTE → FACCAO`: cidades maiores concentram mercado consumidor de drogas, logística (portos, rodovias, aeroportos) e presídios, onde as facções se organizam.
- `FRONT → FACCAO`: estados de fronteira estão nas rotas da cocaína vinda da Bolívia, do Peru e da Colômbia.
- `FACCAO → TXH, ESC, TXT`: a disputa por território e as execuções elevam homicídios, tentativas e mortes sem causa esclarecida.

**FACCAO é um colisor** (PORTE → FACCAO ← FRONT), e é ao mesmo tempo a causa comum dos três sintomas. Com essa rede fazemos o raciocínio **diagnóstico**: observamos porte, estado e sintomas e inferimos P(FACCAO | evidências).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


In [2]:
VARS = ["PORTE", "FRONT", "FACCAO", "TXH", "ESC", "TXT"]
PAIS = {"PORTE": [], "FRONT": [], "FACCAO": ["PORTE", "FRONT"],
        "TXH": ["FACCAO"], "ESC": ["FACCAO"], "TXT": ["FACCAO"]}
dados = d[["PORTE", "FRONT", "ALVO", "TXH", "ESC", "TXT"]].rename(columns={"ALVO": "FACCAO"}).astype(int)
VALORES = {v: sorted(dados[v].unique()) for v in VARS}

## 1. CPTs estimadas dos dados (máxima verossimilhança)

In [3]:
CPT = {}
for v in VARS:
    pais = PAIS[v]
    CPT[v] = dados[v].value_counts(normalize=True).sort_index() if not pais else \
             pd.crosstab([dados[p] for p in pais], dados[v], normalize="index")
    print(f"\nP({v}{' | ' + ', '.join(pais) if pais else ''})"); print(CPT[v].round(4))


P(PORTE)
PORTE
0    0.3304
1    0.5384
2    0.1312
Name: proportion, dtype: float64

P(FRONT)
FRONT
0    0.4406
1    0.5594
Name: proportion, dtype: float64

P(FACCAO | PORTE, FRONT)
FACCAO            0       1
PORTE FRONT                
0     0      0.9869  0.0131
      1      0.7807  0.2193
1     0      0.7955  0.2045
      1      0.5792  0.4208
2     0      0.2963  0.7037
      1      0.1392  0.8608

P(TXH | FACCAO)
TXH          0       1       2
FACCAO                        
0       0.3898  0.3279  0.2823
1       0.2162  0.3436  0.4402

P(ESC | FACCAO)
ESC          0       1
FACCAO                
0       0.4080  0.5920
1       0.2046  0.7954

P(TXT | FACCAO)
TXT          0       1      2
FACCAO                       
0       0.3825  0.3115  0.306
1       0.2317  0.3784  0.390


## 2. Economia de parâmetros

In [4]:
card = {v: len(VALORES[v]) for v in VARS}
p_conj = int(np.prod(list(card.values()))) - 1
p_rede = sum((card[v] - 1) * int(np.prod([card[p] for p in PAIS[v]] or [1])) for v in VARS)
print(f"Conjunta completa: {p_conj} parâmetros | Rede: {p_rede} | Economia: {p_conj - p_rede} ({1 - p_rede / p_conj:.0%})")

Conjunta completa: 215 parâmetros | Rede: 19 | Economia: 196 (91%)


## 3. Inferência por enumeração (do zero)

$$P(X \mid e) = \alpha \sum_{y} \prod_i P(x_i \mid pais(X_i))$$

In [5]:
import itertools

def prob(v, valor, atrib):
    pais = PAIS[v]
    if not pais:
        return CPT[v].get(valor, 0.0)
    chave = tuple(atrib[p] for p in pais)
    chave = chave[0] if len(chave) == 1 else chave
    try:
        return CPT[v].loc[chave, valor]
    except KeyError:
        return 0.0

def conjunta(atrib):
    return np.prod([prob(v, atrib[v], atrib) for v in VARS])

def enumeracao(consulta, evidencia):
    ocultas = [v for v in VARS if v != consulta and v not in evidencia]
    dist = {}
    for q in VALORES[consulta]:
        s = 0.0
        for combo in itertools.product(*[VALORES[h] for h in ocultas]):
            atrib = dict(evidencia); atrib[consulta] = q; atrib.update(zip(ocultas, combo))
            s += conjunta(atrib)
        dist[int(q)] = s
    z = sum(dist.values())
    return {k: float(v / z) for k, v in dist.items()}

consultas = [
    ("FACCAO", {}),
    ("FACCAO", {"PORTE": 2}),
    ("FACCAO", {"PORTE": 0}),
    ("FACCAO", {"PORTE": 0, "FRONT": 1, "TXH": 2, "ESC": 1, "TXT": 2}),
    ("FACCAO", {"PORTE": 2, "FRONT": 0, "TXH": 0, "ESC": 0, "TXT": 0}),
    ("FACCAO", {"TXH": 2, "ESC": 1, "TXT": 2}),
    ("TXH", {"FACCAO": 1}),
]
res_enum = {str(e) + q: enumeracao(q, e) for q, e in consultas}
for q, e in consultas:
    print(f"P({q} | {e}) = { {k: round(v, 4) for k, v in res_enum[str(e) + q].items()} }")

P(FACCAO | {}) = {0: 0.6785, 1: 0.3215}
P(FACCAO | {'PORTE': 2}) = {0: 0.2084, 1: 0.7916}
P(FACCAO | {'PORTE': 0}) = {0: 0.8716, 1: 0.1284}
P(FACCAO | {'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, 'TXT': 2}) = {0: 0.5715, 1: 0.4285}
P(FACCAO | {'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, 'TXT': 0}) = {0: 0.7142, 1: 0.2858}
P(FACCAO | {'TXH': 2, 'ESC': 1, 'TXT': 2}) = {0: 0.4415, 1: 0.5585}
P(TXH | {'FACCAO': 1}) = {0: 0.2162, 1: 0.3436, 2: 0.4402}


## 4. Comparação com pgmpy (VariableElimination)

In [6]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination
arestas = [(p, v) for v in VARS for p in PAIS[v]]
modelo = DiscreteBayesianNetwork(arestas)
modelo.fit(dados)            # estimador padrão: máxima verossimilhança
ve = VariableElimination(modelo)
linhas = []
for q, e in consultas:
    r = ve.query([q], evidence=e, show_progress=False)
    for i, val in enumerate(r.state_names[q]):
        linhas.append([q, str(e), int(val), res_enum[str(e) + q][int(val)], float(r.values[i])])
cmp = pd.DataFrame(linhas, columns=["consulta", "evidência", "valor", "enumeração (própria)", "pgmpy"])
cmp["diferença"] = (cmp["enumeração (própria)"] - cmp["pgmpy"]).abs()
print("Maior diferença:", cmp["diferença"].max())
cmp.round(5)

Maior diferença: 1.1102230246251565e-16


,consulta,evidência,valor,enumeração (própria),pgmpy,diferença
0,FACCAO,{},0,0.67845,0.67845,0.0
1,FACCAO,{},1,0.32155,0.32155,0.0
2,FACCAO,{'PORTE': 2},0,0.20844,0.20844,0.0
3,FACCAO,{'PORTE': 2},1,0.79156,0.79156,0.0
4,FACCAO,{'PORTE': 0},0,0.87156,0.87156,0.0
5,FACCAO,{'PORTE': 0},1,0.12844,0.12844,0.0
6,FACCAO,"{'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, '...",0,0.57150,0.57150,0.0
7,FACCAO,"{'PORTE': 0, 'FRONT': 1, 'TXH': 2, 'ESC': 1, '...",1,0.42850,0.42850,0.0
8,FACCAO,"{'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, '...",0,0.71421,0.71421,0.0
9,FACCAO,"{'PORTE': 2, 'FRONT': 0, 'TXH': 0, 'ESC': 0, '...",1,0.28579,0.28579,0.0


## 5. d-separação

1. **Causa comum (garfo):** `TXH ← FACCAO → ESC`. Dado FACCAO, os sintomas são independentes entre si.
2. **Colisor:** `PORTE → FACCAO ← FRONT`. Porte e fronteira são independentes **até observarmos FACCAO**. Saber que um município pequeno tem facção torna mais provável que ele esteja num estado de fronteira (*explaining away*).

In [7]:
print("TXH ⊥ ESC    | FACCAO ?   ", not modelo.is_dconnected("TXH", "ESC", observed=["FACCAO"]))
print("PORTE ⊥ FRONT ?           ", not modelo.is_dconnected("PORTE", "FRONT", observed=[]))
print("PORTE ⊥ FRONT | FACCAO ?  ", not modelo.is_dconnected("PORTE", "FRONT", observed=["FACCAO"]))
print()
for p in (0, 2):
    print(f"P(FRONT=1 | PORTE={p})              = {enumeracao('FRONT', {'PORTE': p})[1]:.4f}")
for p in (0, 2):
    print(f"P(FRONT=1 | PORTE={p}, FACCAO=1)    = {enumeracao('FRONT', {'PORTE': p, 'FACCAO': 1})[1]:.4f}")
print("\nConferência nos dados: correlação entre PORTE e FRONT =", round(dados["PORTE"].corr(dados["FRONT"]), 3))

TXH ⊥ ESC    | FACCAO ?    True
PORTE ⊥ FRONT ?            True
PORTE ⊥ FRONT | FACCAO ?   False

P(FRONT=1 | PORTE=0)              = 0.5594
P(FRONT=1 | PORTE=2)              = 0.5594
P(FRONT=1 | PORTE=0, FACCAO=1)    = 0.9552
P(FRONT=1 | PORTE=2, FACCAO=1)    = 0.6083

Conferência nos dados: correlação entre PORTE e FRONT = 0.211


**Leitura.** Sem observar FACCAO, o porte não muda a crença sobre a fronteira (os dois valores são iguais). Quando sabemos que o município **tem facção**, ser **pequeno** aumenta a probabilidade de estar num estado de fronteira: uma facção num município pequeno é melhor "explicada" pela rota do tráfico do que pelo tamanho. A última linha confere a suposição do modelo nos dados: porte e fronteira têm correlação fraca (cerca de 0,2), então a independência entre eles é uma aproximação razoável, mas não exata. Registramos isso como limitação.

## 6. A rede como classificador diagnóstico (5-fold)

In [8]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from utils import tercis_do_treino
ev_cols = ["PORTE", "FRONT", "TXH", "ESC", "TXT"]

def p_fora(dados_m, divisoes):
    """P(FACCAO=1) de cada município pela rede treinada sem ele.
    Os tercis de TXH e TXT são refeitos com os cortes do treino de cada divisão."""
    p = np.full(len(dados_m), np.nan)
    for tr, te in divisoes:
        Xf = tercis_do_treino(d, dados_m, tr)
        m = DiscreteBayesianNetwork(arestas); m.fit(Xf.iloc[tr]); inf = VariableElimination(m)
        cache = {}
        for i, r in zip(te, Xf.iloc[te][ev_cols].values):
            key = tuple(int(v) for v in r)
            if key not in cache:
                q = inf.query(["FACCAO"], evidence=dict(zip(ev_cols, key)), show_progress=False)
                cache[key] = float(q.values[list(q.state_names["FACCAO"]).index(1)])
            p[i] = cache[key]
    return p

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
y = dados["FACCAO"].values
dobras = list(cv.split(dados, y))
p_oof = p_fora(dados, dobras)   # P(FACCAO=1) de cada município, estimada sem ele no treino
acc, bal, auc = [], [], []
for _, te in dobras:
    p = (p_oof[te] >= 0.5).astype(int)
    acc.append(accuracy_score(y[te], p)); bal.append(balanced_accuracy_score(y[te], p)); auc.append(roc_auc_score(y[te], p_oof[te]))
print(f"Rede bayesiana (limiar 0,5): acurácia={np.mean(acc):.3f}  balanceada={np.mean(bal):.3f}  AUC={np.mean(auc):.3f}")

Rede bayesiana (limiar 0,5): acurácia=0.761  balanceada=0.713  AUC=0.810


## 7. Probabilidades para a Etapa 6 (fora da dobra)

A Etapa 6 usa P(facção) de cada município como entrada do sistema fuzzy. Se essa probabilidade viesse de uma rede ajustada com **todos** os municípios, cada um teria ajudado a estimar a própria probabilidade, e a conferência com o Cartografias 2024 ficaria otimista. Por isso gravamos a probabilidade **fora da dobra** (`p_faccao`): cada município recebe a P(facção) da rede treinada nas outras 4 dobras da validação cruzada acima. A versão dentro da amostra fica em `p_faccao_amostra`, só para comparação.

In [9]:
combos = dados[ev_cols].drop_duplicates()
mapa = {tuple(int(v) for v in r): enumeracao("FACCAO", dict(zip(ev_cols, map(int, r))))[1] for r in combos.values}
d["p_faccao_amostra"] = [mapa[tuple(int(v) for v in r)] for r in dados[ev_cols].values]
d["p_faccao"] = p_oof
print(f"AUC (2024) dentro da amostra: {roc_auc_score(y, d['p_faccao_amostra']):.3f}  |  fora da dobra: {roc_auc_score(y, d['p_faccao']):.3f}")
print("Maior diferença entre as duas probabilidades:", round((d["p_faccao"] - d["p_faccao_amostra"]).abs().max(), 4))
d[["uf", "municipio", "cod_ibge", "pop_2022"] + ev_cols + ["variacao_hom", "p_faccao", "p_faccao_amostra", "ALVO", "REGIME",
   "ALVO_2025", "REGIME_2025", "ENTRADA_2025"]].to_csv("dados/probabilidades_bayesianas.csv", index=False)
print(d["p_faccao"].describe().round(4))

AUC (2024) dentro da amostra: 0.817  |  fora da dobra: 0.809
Maior diferença entre as duas probabilidades: 0.1647
count    808.0000
mean       0.3284
std        0.2902
min        0.0012
25%        0.0748
50%        0.2441
75%        0.5753
max        0.9603
Name: p_faccao, dtype: float64


## 8. Validação geográfica: deixando um estado de fora

Na validação cruzada comum, as dobras misturam os estados: o modelo sempre vê, no treino, municípios do mesmo estado do município testado. Aqui treinamos com 8 estados e testamos no 9º, repetindo para cada um. É um teste mais exigente: mostra se o modelo serve para um estado que ele nunca viu.

In [10]:
ufs = sorted(d["UF"].unique())
por_uf = [(np.where(d["UF"] != u)[0], np.where(d["UF"] == u)[0]) for u in ufs]
p_uf = p_fora(dados, por_uf)
sem = (d["ALVO"] == 0).values
y25 = d["ALVO_2025"].values
print(f"AUC 2024 (todos os municípios juntos): 5-fold={roc_auc_score(y, p_oof):.3f}  |  estado de fora={roc_auc_score(y, p_uf):.3f}")
print(f"AUC da entrada em 2025:                5-fold={roc_auc_score(y25[sem], p_oof[sem]):.3f}  |  estado de fora={roc_auc_score(y25[sem], p_uf[sem]):.3f}")
linhas = []
for u, (_, te) in zip(ufs, por_uf):
    yt = y[te]
    linhas.append([u, len(te), yt.mean(), p_uf[te].mean(), roc_auc_score(yt, p_uf[te]) if 0 < yt.mean() < 1 else np.nan])
pd.DataFrame(linhas, columns=["UF de teste", "municípios", "com facção 2024", "P média prevista", "AUC"]).round(3)

AUC 2024 (todos os municípios juntos): 5-fold=0.809  |  estado de fora=0.800
AUC da entrada em 2025:                5-fold=0.764  |  estado de fora=0.770


,UF de teste,municípios,com facção 2024,P média prevista,AUC
0,AC,22,1.000,0.465,NaN
1,AM,62,0.323,0.324,0.651
2,AP,16,0.312,0.507,0.845
3,MA,217,0.221,0.205,0.790
4,MT,141,0.298,0.507,0.746
5,PA,144,0.507,0.528,0.778
6,RO,52,0.500,0.427,0.581
7,RR,15,0.933,0.469,0.643
8,TO,139,0.065,0.110,0.938


**Leitura.** No conjunto, o modelo quase não perde ao testar num estado que não viu (AUC 0,80 contra 0,81). Mas a tabela por estado mostra o que a média esconde:

- O modelo **ordena** bem os municípios em TO, AP, MA e PA (AUC de 0,78 a 0,94), mas mal em RO (0,58), RR (0,64) e AM (0,65).
- Ele **erra o nível** onde a facção é quase universal: no Acre (100% com facção) e em Roraima (93%), a P(facção) média prevista é cerca de 0,47. A única informação sobre o estado que a rede recebe é `FRONT`, que separa só MA e TO dos demais.

Conclusão: a rede serve para **ordenar** municípios dentro de um estado conhecido, não para dizer o **nível** de facção num estado novo. Para isso ela precisaria de alguma informação sobre a prevalência no estado, ou de uma variável de fronteira por município (faixa de fronteira do IBGE) em vez de por estado.

## 9. Sensibilidade ao rótulo

O rótulo do Cartografias tem ruído: "sem registro" não é ausência confirmada, e parte das mudanças entre a 3ª e a 4ª edição vem da melhoria do método. Não dá para corrigir o rótulo, mas dá para medir se as conclusões dependem dele:

1. **Sem os municípios que "saíram" da lista** (facção em 2024, sem registro em 2025): se forem erros de registro, a rede deve vê-los com perfil de município sem facção.
2. **Rótulo "facção em 2024 OU em 2025"**: recupera positivos que a 3ª edição pode ter deixado de identificar.

In [11]:
saiu = (d["ALVO"] == 1).values & (y25 == 0)
ficou = (d["ALVO"] == 1).values & (y25 == 1)
uniao = dados.assign(FACCAO=((d["ALVO"] == 1) | (d["ALVO_2025"] == 1)).astype(int).values)
p_uniao = p_fora(uniao, list(cv.split(uniao, uniao["FACCAO"])))
print(f"Rótulo original (2024):                          AUC = {roc_auc_score(y, p_oof):.3f}")
print(f"Sem os {saiu.sum()} municípios que saíram da lista:      AUC = {roc_auc_score(y[~saiu], p_oof[~saiu]):.3f}")
print(f"Rótulo 'facção em 2024 OU 2025' ({uniao['FACCAO'].mean():.1%} positivos): AUC = {roc_auc_score(uniao['FACCAO'], p_uniao):.3f}")
print()
print(f"P(facção) média prevista: saíram da lista = {p_oof[saiu].mean():.3f} | com facção nas duas edições = {p_oof[ficou].mean():.3f}"
      f" | sem facção em 2024 = {p_oof[sem].mean():.3f}")

Rótulo original (2024):                          AUC = 0.809
Sem os 45 municípios que saíram da lista:      AUC = 0.832
Rótulo 'facção em 2024 OU 2025' (48.0% positivos): AUC = 0.823

P(facção) média prevista: saíram da lista = 0.395 | com facção nas duas edições = 0.576 | sem facção em 2024 = 0.226


**Leitura.** As conclusões não dependem do ruído do rótulo; ao contrário, melhoram quando ele é atenuado:

- Sem os 45 municípios que "saíram" da lista, a AUC sobe de 0,81 para 0,83. A rede já via esses municípios com probabilidade intermediária (0,40), mais perto de quem não tem facção (0,23) do que de quem tem nas duas edições (0,58). Isso é coerente com a hipótese de que parte deles foi registro ou mudança de método, e não saída real da facção.
- Com o rótulo "facção em 2024 ou 2025", a AUC sobe para 0,82: parte dos "falsos positivos" de 2024 eram municípios em que a facção só foi identificada na edição seguinte.

Por isso, no relatório, falamos em facção **registrada**, e não presente, e tratamos municípios com P(facção) alta e sem registro (como Oriximiná, PA) como candidatos a verificação, não como erros do modelo.